# Assignment: Building a Modular Data Sanitization & Exploration Engine

### Background
In real-world data science, 80% of the work is spent cleaning and exploring data. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to build a **Reusable Python Class** named `DataInspector` and a supporting `PlottingMethods` class that can be imported into Google Colab to automate these tasks.

### The Objective
Develop an end-to-end tool for CSV data ingestion, advanced cleaning, feature engineering preparation, and high-level statistical visualization.

### Technical Requirements

#### 1. Data Ingestion & Sanitization
* **Colab Integration**: Implement `upload_data()` to handle local file uploads.
* **Garbage String Handling**: Automatically recognize and convert strings like `'?'`, `'n/a'`, `'NULL'`, and `' '` into actual `NaN` values.
* **Auto-Type Correction**: Force-convert columns to numeric types if the conversion does not result in an entirely null column.

#### 2. Structural Analysis & Cleaning
* **Data Summary**: Provide a method to display row/column counts, a preview of the first 20 rows, and a breakdown of numerical vs. categorical columns.
* **Intelligent Imputation**: Create a `handle_missing_values()` method supporting multiple strategies: `mean`, `median`, `mode`, or a `constant` value.
* **Duplicate & Outlier Management**:
    * Implement `remove_duplicates()` to prune exact row matches.
    * Develop an **IQR-based** outlier detection system (`handle_outliers`) that allows users to flag or automatically delete rows based on specific columns.
* **Targeted Deletion**: Implement interactive methods (`delete_rows`, `delete_columns`) that accept comma-separated user input to prune the dataset.

#### 3. Feature Engineering Preparation (Normalization)
* **Numeric Scaling**: Implement `extract_normalized_numeric_data()` supporting `minmax`, `standard` (Z-score), and `robust` (IQR-based) scaling.
* **Categorical Encoding**: Implement `extract_normalized_categorical_data()` supporting `onehot`, `ordinal`, and `uniform` (scaled 0-1) encoding.
* **Dataset Merging**: Provide a method to create a unified DataFrame containing original numeric data alongside encoded categorical data.

#### 4. Advanced Interactive Visualization (Plotly)
* **Univariate Subplots**: For numeric columns, generate a 3-panel subplot: **Horizontal Violin/Box**, **Scatter Plot** (Index vs Value), and **Histogram**.
* **Smart Relationships**: Build a `plot_relationship()` tool that detects types and chooses the correct chart:
    * **Num-Num**: Scatter with OLS Trendline.
    * **Cat-Num**: Box plot with all data points.
    * **Cat-Cat**: Grouped Bar chart.
* **Categorical Frequency**: Create bar charts displaying both raw counts and percentage labels.

#### 5. Deep Statistical Insights
* **Unified Heatmap**: Develop `plot_all_associations_heatmap()` to visualize relationships across *all* data types:
    * **Numeric-Numeric**: Pearson’s $r$.
    * **Categorical-Categorical**: Cramér’s $V$.
    * **Mixed (Num-Cat)**: Point-Biserial correlation or Eta (via ANOVA).

#### 6. Custom Modular Plotting
Implement a separate `PlottingMethods` class to handle granular chart generation (Bar, Pie, Histogram) that returns HTML-wrapped figures for flexible embedding.

### Submission Criteria
1.  **Object-Oriented Design**: All logic must be encapsulated within the `DataInspector` and `PlottingMethods` classes.
2.  **Clean Code**: Every method must include descriptive **Docstrings** and handle empty/None data gracefully.
3.  **Real-world Testing**: Demonstrate the tool using a dataset (e.g., Titanic) by performing a full flow: Upload $\rightarrow$ Impute $\rightarrow$ Normalize $\rightarrow$ Visualize Associations.

## Solution Section

Below are the complete implementations of the `PlottingMethods` and `DataInspector` classes satisfying all technical specifications, followed by an end-to-end analytical validation using the Titanic dataset.

In [ ]:
import io
import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
try:
    from google.colab import files
    colab_available = True
except ImportError:
    colab_available = False

class PlottingMethods:
    """
    Custom Modular Plotting Class designed to handle granular chart generation.
    All methods return HTML-wrapped figures for seamless cross-environment rendering.
    """
    
    @staticmethod
    def generate_bar_chart(df, column, title=None):
        """
        Generates a standard frequency bar chart with raw counts and percentages.
        """
        if df is None or column not in df.columns:
            return "<p>Invalid DataFrame or Column</p>"
        counts = df[column].value_counts(dropna=False).reset_index()
        counts.columns = [column, 'Count']
        total = counts['Count'].sum()
        counts['Percentage'] = (counts['Count'] / total * 100).round(2)
        
        fig = px.bar(counts, x=column, y='Count', 
                     text=counts.apply(lambda r: f"{r['Count']} ({r['Percentage']}%)", axis=1),
                     title=title or f"Frequency Distribution of {column}",
                     labels={'Count': 'Frequency', column: str(column)})
        fig.update_traces(textposition='outside')
        fig.update_layout(template='plotly_white')
        return fig.to_html(full_html=False, include_plotlyjs='cdn')

    @staticmethod
    def generate_pie_chart(df, column, title=None):
        """
        Generates a standard categorical percentage allocation pie chart.
        """
        if df is None or column not in df.columns:
            return "<p>Invalid DataFrame or Column</p>"
        counts = df[column].value_counts(dropna=False).reset_index()
        counts.columns = [column, 'Count']
        
        fig = px.pie(counts, names=column, values='Count',
                     title=title or f"Proportional Allocation of {column}")
        fig.update_traces(textinfo='percent+label')
        return fig.to_html(full_html=False, include_plotlyjs='cdn')

    @staticmethod
    def generate_histogram(df, column, bins=30, title=None):
        """
        Generates a continuous quantitative distribution histogram.
        """
        if df is None or column not in df.columns:
            return "<p>Invalid DataFrame or Column</p>"
        fig = px.histogram(df, x=column, nbins=bins,
                           title=title or f"Continuous Distribution of {column}",
                           labels={column: str(column), 'count': 'Frequency'})
        fig.update_layout(template='plotly_white')
        return fig.to_html(full_html=False, include_plotlyjs='cdn')


class DataInspector:
    """
    An unified, automated Data Sanitization, Structural Analysis, Feature Normalization,
    and Statistical Exploration Engine.
    """
    def __init__(self, df=None):
        self.df = df.copy() if df is not None else None
        if self.df is not None:
            self._sanitize_garbage_strings()
            self._auto_type_correction()

    def upload_data(self):
        """
        Handles local CSV uploads via Google Colab files API. Fallback provided for external runtimes.
        """
        if colab_available:
            print("Please upload your target CSV file:")
            uploaded = files.upload()
            if uploaded:
                file_name = list(uploaded.keys())[0]
                self.df = pd.read_csv(io.BytesIO(uploaded[file_name]))
                print(f"Successfully ingested {file_name}.")
                self._sanitize_garbage_strings()
                self._auto_type_correction()
            else:
                print("Upload sequence aborted by user.")
        else:
            print("Google Colab utilities are not accessible in this system context.")

    def _sanitize_garbage_strings(self):
        """
        Maps dirty text strings/garbage indicators to proper IEEE floating-point NaNs.
        """
        if self.df is None: return
        garbage_patterns = ['?', 'n/a', 'N/A', 'null', 'NULL', ' ', '']
        # Clean cell-level whitespace prior to comparison
        self.df = self.df.applymap(lambda val: np.nan if str(val).strip() in garbage_patterns else val)

    def _auto_type_correction(self):
        """
        Attempts conversion of categorical/object sequences to numeric datatypes safely.
        """
        if self.df is None: return
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                try:
                    converted = pd.to_numeric(self.df[col], errors='coerce')
                    if not converted.isna().all():
                        self.df[col] = converted
                except Exception:
                    pass

    def display_summary(self):
        """
        Displays dimensions, structural previews, and type segregation properties.
        """
        if self.df is None:
            print("Engine currently empty. Load data prior to analytical reporting.")
            return
        print("="*60)
        print(f"DATASET DIMENSIONS: Rows: {self.df.shape[0]} | Columns: {self.df.shape[1]}")
        print("="*60)
        
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()
        
        print(f"Numerical Subspace ({len(numeric_cols)} cols): {numeric_cols}")
        print(f"Categorical Subspace ({len(categorical_cols)} cols): {categorical_cols}")
        print("\n--- COMPREHENSIVE NULL VALUE MATRIX ---")
        null_counts = self.df.isna().sum()
        for c in self.df.columns:
            print(f"Column: {c:<20} | Type: {str(self.df[c].dtype):<10} | Nulls: {null_counts[c]:<6} ({ (null_counts[c]/len(self.df)*100):.2f}%)")
        
        print("\n--- FIRST 20 ROWS DATA PREVIEW ---")
        display(self.df.head(20))
        print("="*60)

    def handle_missing_values(self, columns, strategy='mean', constant_value=None):
        """
        Implements variable imputation profiles for designated targets.
        """
        if self.df is None: return
        if isinstance(columns, str): columns = [columns]
        
        for col in columns:
            if col not in self.df.columns: continue
            if strategy == 'mean' and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col].fillna(self.df[col].mean(), inplace=True)
            elif strategy == 'median' and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col].fillna(self.df[col].median(), inplace=True)
            elif strategy == 'mode':
                mode_series = self.df[col].mode()
                if not mode_series.empty:
                    self.df[col].fillna(mode_series[0], inplace=True)
            elif strategy == 'constant':
                self.df[col].fillna(constant_value, inplace=True)

    def remove_duplicates(self):
        """
        Performs row-wise identical match purging.
        """
        if self.df is None: return
        init_rows = self.df.shape[0]
        self.df.drop_duplicates(inplace=True)
        print(f"Purged {init_rows - self.df.shape[0]} duplicate records. Retained rows: {self.df.shape[0]}")

    def handle_outliers(self, columns, action='flag', threshold=1.5):
        """
        IQR-driven anomaly management engine. Supports flagging or active deletion.
        """
        if self.df is None: return
        if isinstance(columns, str): columns = [columns]
        
        rows_to_drop = set()
        for col in columns:
            if col not in self.df.columns or not pd.api.types.is_numeric_dtype(self.df[col]):
                continue
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR
            
            outliers_idx = self.df[(self.df[col] < lower_bound) | (self.df[col] > upper_bound)].index
            
            if action == 'flag':
                self.df[f'{col}_outlier'] = 0
                self.df.loc[outliers_idx, f'{col}_outlier'] = 1
                print(f"Flagged {len(outliers_idx)} outliers inside vector '{col}'. Created feature '{col}_outlier'")
            elif action == 'delete':
                rows_to_drop.update(outliers_idx)
        
        if action == 'delete' and rows_to_drop:
            self.df.drop(index=list(rows_to_drop), inplace=True)
            print(f"Successfully pruned {len(rows_to_drop)} anomalous row vectors across specified spaces.")

    def delete_rows(self, row_indices_str):
        """
        Prunes target index profiles via discrete comma-delimited strings.
        """
        if self.df is None: return
        try:
            indices = [int(idx.strip()) for idx in row_indices_str.split(',') if idx.strip()]
            valid_indices = [idx for idx in indices if idx in self.df.index]
            self.df.drop(index=valid_indices, inplace=True)
            print(f"Pruned row profiles matching indicators: {valid_indices}")
        except Exception as e:
            print(f"Malformed entry sequence provided for row elimination: {e}")

    def delete_columns(self, column_names_str):
        """
        Drops target features via discrete comma-delimited inputs.
        """
        if self.df is None: return
        columns = [col.strip() for col in column_names_str.split(',') if col.strip()]
        valid_cols = [c for c in columns if c in self.df.columns]
        self.df.drop(columns=valid_cols, inplace=True)
        print(f"Dropped column projections: {valid_cols}")

    def extract_normalized_numeric_data(self, columns, method='standard'):
        """
        Transforms numeric vectors using minmax, standard Z-Score, or robust scaling architectures.
        """
        if self.df is None: return pd.DataFrame()
        if isinstance(columns, str): columns = [columns]
        
        scaled_df = pd.DataFrame(index=self.df.index)
        for col in columns:
            if col not in self.df.columns or not pd.api.types.is_numeric_dtype(self.df[col]):
                continue
            series = self.df[col]
            if method == 'minmax':
                denom = (series.max() - series.min())
                scaled_df[f'{col}_minmax'] = (series - series.min()) / denom if denom != 0 else 0.0
            elif method == 'standard':
                std = series.std()
                scaled_df[f'{col}_standard'] = (series - series.mean()) / std if std != 0 else 0.0
            elif method == 'robust':
                q25, q75 = series.quantile(0.25), series.quantile(0.75)
                iqr = q75 - q25
                scaled_df[f'{col}_robust'] = (series - series.median()) / iqr if iqr != 0 else 0.0
        return scaled_df

    def extract_normalized_categorical_data(self, columns, method='onehot'):
        """
        Encodes nominal/ordinal variables into standardized operational arrays.
        """
        if self.df is None: return pd.DataFrame()
        if isinstance(columns, str): columns = [columns]
        
        encoded_df = pd.DataFrame(index=self.df.index)
        for col in columns:
            if col not in self.df.columns: continue
            series = self.df[col].astype(str)
            
            if method == 'onehot':
                dummies = pd.get_dummies(series, prefix=col, drop_first=False).astype(int)
                encoded_df = pd.concat([encoded_df, dummies], axis=1)
            elif method in ['ordinal', 'uniform']:
                cats = sorted(series.unique())
                mapping = {cat: idx for idx, cat in enumerate(cats)}
                encoded_series = series.map(mapping)
                if method == 'uniform' and len(cats) > 1:
                    encoded_df[f'{col}_uniform'] = encoded_series / (len(cats) - 1)
                else:
                    encoded_df[f'{col}_ordinal'] = encoded_series
        return encoded_df

    def create_unified_dataset(self, numeric_cols, categorical_cols, num_method='standard', cat_method='onehot'):
        """
        Unifies original numeric metrics structurally alongside engineered categorical representations.
        """
        if self.df is None: return None
        orig_numeric = self.df[numeric_cols].copy()
        norm_cat = self.extract_normalized_categorical_data(categorical_cols, method=cat_method)
        return pd.concat([orig_numeric, norm_cat], axis=1)

    def generate_univariate_subplots(self, column):
        """
        Constructs interactive 3-panel subplots (Violin/Box, Index Scatter, Histogram) for metrics.
        """
        if self.df is None or column not in self.df.columns:
            print("Target feature parameters unreachable.")
            return
        
        valid_data = self.df[column].dropna()
        fig = make_subplots(
            rows=3, cols=1,
            subplot_titles=(f"Violin & Box Plot Profile: {column}", 
                            f"Sequential Index Distribution Plot: {column}", 
                            f"Frequency Density Histogram: {column}"),
            vertical_spacing=0.12
        )
        
        fig.add_trace(go.Violin(x=valid_data, box_visible=True, meanline_visible=True, name=column, marker_color='#1f77b4'), row=1, col=1)
        fig.add_trace(go.Scatter(x=valid_data.index, y=valid_data, mode='markers', marker=dict(color='#ff7f0e', opacity=0.6), name='Value'), row=2, col=1)
        fig.add_trace(go.Histogram(x=valid_data, nbinsx=30, marker_color='#2ca02c', name='Count'), row=3, col=1)
        
        fig.update_layout(height=800, title_text=f"Univariate Comprehensive Analysis Matrix: {column}", showlegend=False, template='plotly_white')
        fig.show()

    def plot_relationship(self, col1, col2):
        """
        Automated polymorphic bivariate display chooser evaluating structural space properties.
        """
        if self.df is None or col1 not in self.df.columns or col2 not in self.df.columns:
            return
        
        is_num1 = pd.api.types.is_numeric_dtype(self.df[col1])
        is_num2 = pd.api.types.is_numeric_dtype(self.df[col2])
        
        if is_num1 and is_num2:
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", 
                             title=f"Bivariate Quantitative Relationship: {col1} vs {col2}")
        elif not is_num1 and not is_num2:
            counts = self.df.groupby([col1, col2]).size().reset_index(name='Count')
            fig = px.bar(counts, x=col1, y='Count', color=col2, barmode='group',
                         title=f"Bivariate Categorical Contrast: {col1} by {col2}")
        else:
            num_col = col1 if is_num1 else col2
            cat_col = col2 if is_num1 else col1
            fig = px.box(self.df, x=cat_col, y=num_col, points="all",
                         title=f"Stratified Distribution Matrix: {num_col} segmented by {cat_col}")
            
        fig.update_layout(template='plotly_white')
        fig.show()

    def plot_all_associations_heatmap(self):
        """
        Computes cross-type associations (Pearson's r, Cramér's V, ANOVA Eta) to render a unified matrix.
        """
        if self.df is None: return
        cols = self.df.columns.tolist()
        n = len(cols)
        matrix = np.zeros((n, n))
        
        for i in range(n):
            for j in range(n):
                if i == j:
                    matrix[i, j] = 1.0
                    continue
                c1, c2 = cols[i], cols[j]
                is_num1 = pd.api.types.is_numeric_dtype(self.df[c1])
                is_num2 = pd.api.types.is_numeric_dtype(self.df[c2])
                
                clean_df = self.df[[c1, c2]].dropna()
                if clean_df.empty:
                    matrix[i, j] = 0.0
                    continue
                
                if is_num1 and is_num2:
                    # Quantitative pairing -> Pearson product-moment coefficient
                    r, _ = stats.pearsonr(clean_df[c1], clean_df[c2])
                    matrix[i, j] = round(r, 2) if not np.isnan(r) else 0.0
                elif not is_num1 and not is_num2:
                    # Nominal pairing -> Cramér's V calculation
                    confusion_matrix = pd.crosstab(clean_df[c1], clean_df[c2])
                    chi2 = stats.chi2_contingency(confusion_matrix)[0]
                    n_obs = confusion_matrix.sum().sum()
                    phi2 = chi2 / n_obs
                    r, k = confusion_matrix.shape
                    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n_obs-1))
                    rcorr = r - ((r-1)**2)/(n_obs-1)
                    kcorr = k - ((k-1)**2)/(n_obs-1)
                    if min((kcorr-1), (rcorr-1)) == 0:
                        v = 0.0
                    else:
                        v = np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))
                    matrix[i, j] = round(v, 2) if not np.isnan(v) else 0.0
                else:
                    # Heterogeneous pairing -> Correlation Ratio (Eta)
                    num_c = c1 if is_num1 else c2
                    cat_c = c2 if is_num1 else c1
                    groups = [group[num_c].values for name, group in clean_df.groupby(cat_c)]
                    if len(groups) > 1 and sum(len(g) for g in groups) > len(groups):
                        f_val, _ = stats.f_oneway(*groups)
                        # Approximate correlation ratio via anova space partitions
                        total_ss = np.sum((clean_df[num_c] - clean_df[num_c].mean())**2)
                        if total_ss == 0:
                            eta = 0.0
                        else:
                            between_ss = sum(len(g) * (np.mean(g) - clean_df[num_c].mean())**2 for g in groups)
                            eta = np.sqrt(between_ss / total_ss)
                        matrix[i, j] = round(eta, 2) if not np.isnan(eta) else 0.0
                    else:
                        matrix[i, j] = 0.0
                        
        fig = px.imshow(matrix, x=cols, y=cols, text_auto=True, aspect="auto",
                        color_continuous_scale=px.colors.sequential.Cividis,
                        title="Unified Association Matrix Heatmap (Pearson r / Cramér V / ANOVA Eta)")
        fig.show()


## Real-World Verification Flow Using the Titanic Dataset

In [ ]:
# Ingest Titanic Dataset from public URL repository
titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_raw = pd.read_csv(titanic_url)

# Inject artificial garbage sequence values to showcase handling robust features
df_raw.iloc[5, df_raw.columns.get_loc('Age')] = '?'
df_raw.iloc[10, df_raw.columns.get_loc('Cabin')] = 'NULL'
df_raw.iloc[15, df_raw.columns.get_loc('Fare')] = ' '

print("Engine Instantiation phase initiated...")
inspector = DataInspector(df_raw)

# 1. Structural Analysis Output Reporting
inspector.display_summary()

In [ ]:
# 2. Intelligent Imputation Execution Sequence
print("Applying strategy-based imputations across sparse vectors...")
inspector.handle_missing_values('Age', strategy='median')
inspector.handle_missing_values('Embarked', strategy='mode')
inspector.handle_missing_values('Cabin', strategy='constant', constant_value='Unspecified')

# 3. Redundancy / Duplicate Match Removal Execution
inspector.remove_duplicates()

# 4. IQR Outlier Processing Sequence
inspector.handle_outliers('Fare', action='flag', threshold=1.5)

# Verify integrity after processing steps
inspector.display_summary()

In [ ]:
# 5. Normalization Subspace Compilation Step
numeric_targets = ['Age', 'Fare']
categorical_targets = ['Sex', 'Embarked']

print("Extracting Normalized / Scaled Numerical Matrix Space:")
norm_num = inspector.extract_normalized_numeric_data(numeric_targets, method='standard')
display(norm_num.head())

print("\nExtracting One-Hot Categorical Array Projection Space:")
norm_cat = inspector.extract_normalized_categorical_data(categorical_targets, method='onehot')
display(norm_cat.head())

print("\nCompiling Cohesive Modeling Feature Dataset Tensor:")
unified_df = inspector.create_unified_dataset(numeric_targets, categorical_targets, cat_method='onehot')
display(unified_df.head())

In [ ]:
# 6. Advanced Visualization Matrix Render Generation
print("Rendering Univariate Distribution Subplot for numerical vector 'Age'...")
inspector.generate_univariate_subplots('Age')

print("\nRendering Polymorphic Bivariate Analysis for Sex vs Survived (Cat-Num Relationship)...")
inspector.plot_relationship('Sex', 'Survived')

print("\nGenerating Polymorphic Bivariate Analysis for Age vs Fare (Num-Num Relationship)...")
inspector.plot_relationship('Age', 'Fare')

print("\nDisplaying Unified Heterogeneous System Association Heatmap Matrix...")
# Drop unique IDs and unparsed high-cardinality text before printing correlation maps
inspector.delete_columns("PassengerId,Name,Ticket")
inspector.plot_all_associations_heatmap()

In [ ]:
# 7. Custom Modular Component Output Validation
print("Validating separate HTML module output framework via PlottingMethods backend...")
html_chart = PlottingMethods.generate_bar_chart(inspector.df, 'Pclass', title='Titanic Socioeconomic Ticket Class Share Allocation')
print(f"Successfully exported component as HTML block text. Character length: {len(html_chart)}")